In [1]:
import joblib
import numpy as np
import pandas as pd
import glob
import os
from tensorflow.keras.models import load_model
import keras

# Workaround for Keras quantization_config error
for layer_class in [
    keras.layers.Dense, keras.layers.Dropout, keras.layers.BatchNormalization, 
    keras.layers.Activation, keras.layers.Conv1D, keras.layers.MaxPooling1D, 
    keras.layers.Flatten, keras.layers.LSTM, keras.layers.GRU, keras.layers.Bidirectional
]:
    orig_init = layer_class.__init__
    def make_safe_init(orig):
        def safe_init(self, *args, **kwargs):
            kwargs.pop('quantization_config', None)
            orig(self, *args, **kwargs)
        return safe_init
    layer_class.__init__ = make_safe_init(orig_init)

# ---------------- 1. Load Dataset & Preprocess ----------------
print("Loading dataset...")
# Change this to 'Data Warehouse Multiclass.csv' if that is your dataset name
dataset_filename = 'Multi-Disease_Prediction.csv'
df = pd.read_csv(dataset_filename)

# Select all 24 independent features
X_full = df.iloc[:, :24].copy()

# Encode categorical feature columns
encoders = joblib.load('Label-Encoder.bin')
for column in X_full.select_dtypes(include=['object', 'category']):
    if column in encoders:
        X_full[column] = encoders[column].transform(X_full[column])

# Scale full feature set using saved RobustScaler
scaler = joblib.load('Robust-Scaler.bin')
X_full_scaled = scaler.transform(X_full)

print("Preprocessing done. Generating predictions...\n")

# Create separate dataframes for our two CSV requirements
df_all = df.copy()
df_selected = df.copy()

# Get all model files in the directory
ml_models = glob.glob('*_Model.pkl')
dl_models = glob.glob('*_Model.keras')

# ---------------- 2. Model Loading & Prediction Loop ----------------

# --- ML Predictions ---
for ml_path in ml_models:
    name = ml_path.replace('_Model.pkl', '')
    print(f"Predicting with ML Model: {name}")
    model = joblib.load(ml_path)
    preds = model.predict(X_full_scaled)
    
    # Save prediction to df_all
    df_all[f'{name}_Predicted_Class'] = preds
    
    # If it's CatBoost, save it to df_selected as well
    if name == 'CatBoost-Classifier':
        df_selected[f'{name}_Predicted_Class'] = preds

# --- DL Predictions ---
for dl_path in dl_models:
    name = dl_path.replace('_Model.keras', '')
    print(f"Predicting with DL Model: {name}")
    model = load_model(dl_path)
    
    raw_probabilities = model.predict(X_full_scaled, batch_size=64, verbose=0)
    predicted_classes = np.argmax(raw_probabilities, axis=1)
    
    # Save prediction to df_all
    df_all[f'{name}_Predicted_Class'] = predicted_classes
    
    # If it's CNN-BiGRU, save it to df_selected as well
    if name == 'CNN-BiGRU':
        df_selected[f'{name}_Predicted_Class'] = predicted_classes

# ---------------- 3. Save Final Datasets ----------------
output_all = 'predictions_all_models.csv'
df_all.to_csv(output_all, index=False)
print(f"\nPredictions for ALL models saved successfully to: {output_all}")

output_selected = 'predictions_catboost_cnn_bigru.csv'
df_selected.to_csv(output_selected, index=False)
print(f"Predictions for CatBoost and CNN-BiGRU saved successfully to: {output_selected}")

# ---------------- 4. Display Results ----------------
display(df_selected.head())

Loading dataset...


d:\Predictor_Applications v1.2\Multi-Disease_Predictor v1.0\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Predictor_Applications v1.2\Multi-Disease_Predictor v1.0\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Preprocessing done. Generating predictions...

Predicting with ML Model: AdaBoost-Classifier


d:\Predictor_Applications v1.2\Multi-Disease_Predictor v1.0\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Predictor_Applications v1.2\Multi-Disease_Predictor v1.0\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator AdaBoostClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Predicting with ML Model: Bagging-Classifier


...

d:\Predictor_Applications v1.2\Multi-Disease_Predictor v1.0\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Predictor_Applications v1.2\Multi-Disease_Predictor v1.0\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Predicting with ML Model: Ridge-Classifier


d:\Predictor_Applications v1.2\Multi-Disease_Predictor v1.0\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelBinarizer from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Predictor_Applications v1.2\Multi-Disease_Predictor v1.0\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RidgeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Predictor_Applications v1.2\Multi-Disease_Predictor v1.0\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarn

Predicting with ML Model: SGD-Classifier
Predicting with ML Model: XGBoost-Classifier
Predicting with ML Model: XGBoostRF-Classifier
Predicting with DL Model: CNN-BiGRU
Predicting with DL Model: CNN-BiLSTM
Predicting with DL Model: CNN-GRU
Predicting with DL Model: CNN-LSTM
Predicting with DL Model: DNN

Predictions for ALL models saved successfully to: predictions_all_models.csv
Predictions for CatBoost and CNN-BiGRU saved successfully to: predictions_catboost_cnn_bigru.csv


,gender,smoking,age,bmi,HbA1c_level,glucose,cholesterol,sleep_hours,triglycerides,physical_activity,...,alcohol_intake,salt_intake,heart_rate,hdl,ldl,education_level,employment_status,sublabel,CatBoost-Classifier_Predicted_Class,CNN-BiGRU_Predicted_Class
0,Female,Never,9,19.20,5.8,158.0,214.703704,6.864403,246.703704,High,...,14.777636,8.685304,74.329073,65.651757,129.220447,Primary,Retired,N,7,7
1,Female,Never,3,22.55,6.6,159.0,214.703704,6.864403,246.703704,High,...,14.777636,8.685304,74.329073,65.651757,129.220447,Primary,Retired,N,7,7
2,Female,Never,3,22.89,4.0,126.0,214.703704,6.864403,246.703704,High,...,14.777636,8.685304,74.329073,65.651757,129.220447,Primary,Retired,N,7,7
3,Female,Never,3,23.12,6.2,85.0,214.703704,6.864403,246.703704,High,...,14.777636,8.685304,74.329073,65.651757,129.220447,Primary,Retired,N,7,7
4,Female,Never,4,19.61,5.7,155.0,214.703704,6.864403,246.703704,High,...,14.777636,8.685304,74.329073,65.651757,129.220447,Primary,Retired,N,7,7
